# HAR-RV vs ModernTCN vs ModernTCN + news events

Trains three models on `data/EURUSD-RV.csv` at **h = 1, 5, 22** and prints one
comparison table.

| Model | What it sees |
|---|---|
| **HAR-RV** | daily / weekly / monthly RV averages (OLS, no hyper-parameters) |
| **ModernTCN** | the ln(RV) look-back window only |
| **ModernTCN + events** | the same window **plus** the macro news calendar — past events at the stem, and the *known* future release schedule FiLM-conditioning the head |

All three predict the same target and score the **same test rows**:

$$Y_t^{(h)} \;=\; \ln\!\Big(\tfrac{1}{h}\sum_{k=1}^{h} RV_{t+k}\Big)$$

**Split** — train `year <= 2021`, val `2022-2023`, test `year >= 2024`. HAR-RV folds
val into train (OLS has nothing to tune). Test counts are 647 / 643 / 626 at
h = 1 / 5 / 22 for every model; section 4 asserts this before anything trains.

**Hyper-parameters** are the single set supplied for this run, held fixed across all
three horizons — only `--pred_len` changes. They are *not* per-horizon tuned; see
the closing notes.

**Runtime** — `Runtime -> Change runtime type -> GPU`. Roughly 15-30 min on a T4 for
the full 3 horizons x 5 seeds x 2 deep models. Set `ITR = 1` in section 2 for a
~5 min smoke pass.

## 1 · Setup

In [ ]:
import os, subprocess, sys

REPO   = "https://github.com/Mr0022/ProjectA.git"
BRANCH = "claude/optimistic-mccarthy-6zszk7"
DIR    = "/content/ProjectA"

if not os.path.isdir(DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, DIR], check=True)
else:
    subprocess.run(["git", "-C", DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(DIR)
sys.path.insert(0, DIR)
print("HEAD:", subprocess.run(["git", "log", "--oneline", "-1"],
                              capture_output=True, text=True).stdout.strip())

# --- guard: refuse to run stale code -----------------------------------------
# The aggregated target must be log(mean RV) == logsumexp(ln_RV) - log(h).
import inspect
from utils.rv import forward_log_mean
assert "rolling(h).mean()" in inspect.getsource(forward_log_mean), (
    "utils/rv.py is stale: forward_log_mean must aggregate with mean, not sum.")
print("target convention OK: ln(mean RV)")

In [ ]:
# Colab ships torch / pandas / numpy / statsmodels / scipy / sklearn.
# This only fills gaps on a bare runtime.
import importlib, subprocess, sys
missing = [p for p, m in [("pandas", "pandas"), ("numpy", "numpy"),
                          ("statsmodels", "statsmodels"), ("scipy", "scipy"),
                          ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("dependencies ready" + (f" (installed: {', '.join(missing)})" if missing else ""))

## 2 · Configuration

In [ ]:
EPOCHS    = 40      # max epochs per seed
PATIENCE  = 8       # early-stopping patience on validation loss
ITR       = 5       # seeds per horizon: 2021..2021+ITR-1. Set 1 for a quick smoke pass.
WORKERS   = 0       # DataLoader workers
BATCH     = 256
LR        = 0.0077943332090161695

HORIZONS  = [1, 5, 22]

# One hyper-parameter set, applied unchanged at every horizon. Only --pred_len moves.
HP = dict(
    seq_len      = 35,
    patch_size   = 32,
    patch_stride = 2,
    ffn_ratio    = 2,
    num_blocks   = 1,     # expanded to "1 1 1 1" (ModernTCN's backbone has 4 stages)
    large_size   = 31,
    small_size   = 5,
    dim          = 64,
    dropout      = 0.0,
    head_dropout = 0.0,
)

# --- event-model settings ----------------------------------------------------
EVENT_DIM    = 32          # width of the learned event-type embedding
EVENT_FUSION = "channel"   # 'channel': past events flow through the whole backbone as
                           # their own variable, so ModernTCN's cross-variable ConvFFN
                           # mixes value<->events at every stage. This is what
                           # tune.py/scripts/eventtcn.sh use. 'inject' adds the event
                           # embedding onto the stem feature map instead.
EVENT_FILE   = "events_daily_features.csv"   # built in section 3 from data/events_daily.csv

print(f"{len(HORIZONS)} horizons x {ITR} seed(s) x 2 deep models, up to {EPOCHS} epochs each")
print(f"event embedding: dim {EVENT_DIM}, fusion '{EVENT_FUSION}'")

In [ ]:
import re, subprocess, sys

_MEAN   = re.compile(r"^\s*mean\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s*$", re.M)
_STD    = re.compile(r"^\s*std\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s+([-\d.eE+]+)\s*$", re.M)
_SINGLE = re.compile(r"mse:\s*([-\d.eE+]+),\s*mae:\s*([-\d.eE+]+),"
                     r"\s*rse:\s*([-\d.eE+]+),\s*qlike:\s*([-\d.eE+]+)")
KEYS = ["mse", "mae", "rse", "qlike"]


def run_stream(cmd, show=r"^(train |val |test |Epoch:|>>>>>>> run|mse:|news events|\s*(seed|mean|std)\s|Early)"):
    """Run a command, echo only the interesting lines, return the full output."""
    pat, lines = re.compile(show), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        lines.append(line)
        if pat.search(line):
            print(line.rstrip())
    proc.wait()
    out = "".join(lines)
    if proc.returncode != 0:
        print("\n".join(out.splitlines()[-30:]))
        raise RuntimeError(f"command failed (exit {proc.returncode}): {' '.join(map(str, cmd))}")
    return out


def parse_metrics(out):
    """Return ({metric: mean}, {metric: std}). std is None for a single run."""
    m, s = _MEAN.search(out), _STD.search(out)
    if m:
        mean = {k: float(v) for k, v in zip(KEYS, m.groups())}
        std  = {k: float(v) for k, v in zip(KEYS, s.groups())} if s else None
        return mean, std
    hits = _SINGLE.findall(out)
    if not hits:
        raise RuntimeError("could not find metrics in the run output")
    return {k: float(v) for k, v in zip(KEYS, hits[-1])}, None


def moderntcn_cmd(h, use_events):
    """The run.py command line for one horizon, with or without the event calendar."""
    rep = lambda v: [str(v)] * 4          # ModernTCN's backbone has 4 stages
    cmd = [sys.executable, "run.py", "--is_training", "1",
           "--model_id", f"{'EventTCN' if use_events else 'ModernTCN'}_h{h}",
           "--model", "ModernTCN",
           "--features", "S", "--enc_in", "1", "--dec_in", "1", "--c_out", "1",
           "--aggregate_mean", "--seq_len", str(HP["seq_len"]), "--pred_len", str(h),
           "--patch_size", str(HP["patch_size"]), "--patch_stride", str(HP["patch_stride"]),
           "--ffn_ratio", str(HP["ffn_ratio"]),
           "--num_blocks", *rep(HP["num_blocks"]),
           "--large_size", *rep(HP["large_size"]), "--small_size", *rep(HP["small_size"]),
           "--dims", *rep(HP["dim"]), "--dw_dims", *rep(HP["dim"]),
           "--dropout", str(HP["dropout"]), "--head_dropout", str(HP["head_dropout"]),
           "--revin", "1", "--use_multi_scale", "False",
           "--lradj", "TST", "--pct_start", "0.3",
           "--learning_rate", str(LR), "--batch_size", str(BATCH),
           "--train_epochs", str(EPOCHS), "--patience", str(PATIENCE),
           "--num_workers", str(WORKERS), "--itr", str(ITR)]
    if use_events:
        # --use_events also switches --data custom -> custom_events
        cmd += ["--use_events", "--event_data_path", EVENT_FILE,
                "--event_dim", str(EVENT_DIM), "--event_fusion", EVENT_FUSION]
    return cmd

print("helpers ready")

## 3 · Build the event calendar

`data/events_daily.csv` is the raw calendar — one row per scheduled release
(`Date, Name, Impact, Currency`). `build_event_features.py` turns it into the wide
daily matrix the loader wants: a `date` column, `n_events*` counts (standardised on
the train years) and `evt_*` 0/1 indicators.

Watch the last line: any trading day the calendar does not reach becomes an all-zero
"no events today" row, which is a lie the model will happily learn.

In [ ]:
out = run_stream([sys.executable, "build_event_features.py"], show=r"^\[")

In [ ]:
# Cheap guard before spending 20+ minutes on training: does the file satisfy
# everything Dataset_Custom_Events assumes? (--no-end-to-end skips the training
# stage, which we are about to do for real anyway.)
out = run_stream([sys.executable, "smoke_test_events.py",
                  "--event-path", EVENT_FILE, "--no-end-to-end"],
                 show=r"^(\s+\[FAIL\]|\d+ passed|FAILED)")

## 4 · Split sanity check

Confirms all three models score the **same** test rows before anything is trained.
Look for the `LSTM / ModernTCN (seq_len=35)` lines to match `HAR-RV` at each horizon.

In [ ]:
print(run_stream([sys.executable, "check_splits.py"], show=r".").strip()[-1500:])

## 5 · HAR-RV baseline

OLS on daily / weekly / monthly RV averages. No hyper-parameters, so validation folds
into train. Seconds to run; writes figures and CSVs into `HAR-RV results/`.

In [ ]:
import pandas as pd

run_stream([sys.executable, "HAR_RV_run.py"], show=r"^(  (Train|Test|Set)|\s+(MSE|MAE|QLIKE)\s)")

har = pd.read_csv("HAR-RV results/har_rv_all_metrics.csv")
har = har[har["split"] == "test"].set_index("horizon")[["MSE", "MAE", "QLIKE"]]
print("\nHAR-RV test metrics")
display(har.round(4))

## 6 · ModernTCN (no events)

Each horizon trains `ITR` seeds (2021, 2022, ...); the table reports the mean and the
spread across them.

In [ ]:
import time

mtcn, mtcn_std = {}, {}
for h in HORIZONS:
    print("\n" + "=" * 78)
    print(f"ModernTCN   h = {h}   (seq_len {HP['seq_len']}, {ITR} seed(s))")
    print("=" * 78)
    t0 = time.time()
    mean, std = parse_metrics(run_stream(moderntcn_cmd(h, use_events=False)))
    mtcn[h], mtcn_std[h] = mean, std
    print(f"--> h={h}: MSE {mean['mse']:.4f}  MAE {mean['mae']:.4f}  "
          f"QLIKE {mean['qlike']:.4f}   ({time.time() - t0:.0f}s)")

## 7 · ModernTCN + news events

Identical architecture and hyper-parameters — the only difference is that the event
calendar is switched on (`--use_events`), so any change in the numbers is attributable
to the events rather than to capacity or tuning.

In [ ]:
evtcn, evtcn_std = {}, {}
for h in HORIZONS:
    print("\n" + "=" * 78)
    print(f"ModernTCN + events   h = {h}   (event_dim {EVENT_DIM}, "
          f"fusion {EVENT_FUSION}, {ITR} seed(s))")
    print("=" * 78)
    t0 = time.time()
    mean, std = parse_metrics(run_stream(moderntcn_cmd(h, use_events=True)))
    evtcn[h], evtcn_std[h] = mean, std
    print(f"--> h={h}: MSE {mean['mse']:.4f}  MAE {mean['mae']:.4f}  "
          f"QLIKE {mean['qlike']:.4f}   ({time.time() - t0:.0f}s)")

## 8 · Comparison table

In [ ]:
import numpy as np, pandas as pd

MODELS = ["HAR-RV", "ModernTCN", "ModernTCN + events"]
SRC = {"ModernTCN": (mtcn, mtcn_std), "ModernTCN + events": (evtcn, evtcn_std)}

rows = []
for h in HORIZONS:
    rows.append(dict(model="HAR-RV", horizon=h, seeds=1,
                     MSE=har.loc[h, "MSE"], MAE=har.loc[h, "MAE"], QLIKE=har.loc[h, "QLIKE"],
                     MSE_std=np.nan, MAE_std=np.nan, QLIKE_std=np.nan))
    for name in ["ModernTCN", "ModernTCN + events"]:
        mean, stds = SRC[name][0][h], SRC[name][1][h]
        rows.append(dict(model=name, horizon=h, seeds=ITR,
                         MSE=mean["mse"], MAE=mean["mae"], QLIKE=mean["qlike"],
                         MSE_std=stds["mse"] if stds else np.nan,
                         MAE_std=stds["mae"] if stds else np.nan,
                         QLIKE_std=stds["qlike"] if stds else np.nan))
long = pd.DataFrame(rows)

pd.set_option("display.width", 220)

wide = long.pivot(index="model", columns="horizon", values=["MSE", "MAE", "QLIKE"])
wide = wide[[(m, h) for h in HORIZONS for m in ("MSE", "MAE", "QLIKE")]]
wide.columns = pd.MultiIndex.from_tuples([(f"h={h}", m) for m, h in wide.columns])
wide = wide.reindex(MODELS)

print("Test-set losses — lower is better")
print("target = ln(mean RV); identical test rows (647 / 643 / 626 at h = 1 / 5 / 22)\n")
display(wide.round(4).style.highlight_min(axis=0, props="font-weight:bold;"))

In [ ]:
# Where does each model win? (best per horizon x metric)
winners = pd.DataFrame(
    {f"h={h}": {m: wide.loc[:, (f"h={h}", m)].idxmin() for m in ("MSE", "MAE", "QLIKE")}
     for h in HORIZONS})
print("Best model per horizon x metric\n")
display(winners)

In [ ]:
# Two questions, two deltas. Negative = the first model is better.
vs_har = pd.DataFrame({
    (name, m): [(SRC[name][0][h][m.lower()] / har.loc[h, m] - 1) * 100 for h in HORIZONS]
    for name in ["ModernTCN", "ModernTCN + events"] for m in ["MSE", "MAE", "QLIKE"]
}, index=[f"h={h}" for h in HORIZONS])
print("vs HAR-RV, % change in loss  (negative = deep model beats HAR-RV)\n")
display(vs_har.round(1))

events_effect = pd.DataFrame({
    m: [(evtcn[h][m.lower()] / mtcn[h][m.lower()] - 1) * 100 for h in HORIZONS]
    for m in ["MSE", "MAE", "QLIKE"]
}, index=[f"h={h}" for h in HORIZONS])
print("\nDo the events help? % change vs plain ModernTCN  (negative = events help)\n")
display(events_effect.round(1))

In [ ]:
# Seed spread matters: a gap smaller than the noise is not a result.
print("Per-run detail — std is across seeds; HAR-RV is deterministic OLS\n")
display(long.set_index(["horizon", "model"]).round(4))

print("\nIs the event effect larger than seed noise?")
for h in HORIZONS:
    gap   = mtcn[h]["qlike"] - evtcn[h]["qlike"]          # >0 means events won
    noise = max(mtcn_std[h]["qlike"] if mtcn_std[h] else 0,
                evtcn_std[h]["qlike"] if evtcn_std[h] else 0)
    verdict = "clear" if abs(gap) > 2 * noise else ("marginal" if abs(gap) > noise else "within noise")
    print(f"  h={h:2d}  QLIKE gap {gap:+.4f}   worst seed-std {noise:.4f}   -> {verdict}")

## 9 · Save

In [ ]:
long.to_csv("comparison_three_models.csv", index=False)

md_lines = ["| Model | " + " | ".join(f"h={h} {m}" for h in HORIZONS
                                      for m in ("MSE", "MAE", "QLIKE")) + " |",
            "|" + "---|" * (1 + 3 * len(HORIZONS))]
for model in MODELS:
    cells_ = [model]
    for h in HORIZONS:
        r = long[(long.model == model) & (long.horizon == h)].iloc[0]
        cells_ += [f"{r.MSE:.4f}", f"{r.MAE:.4f}", f"{r.QLIKE:.4f}"]
    md_lines.append("| " + " | ".join(cells_) + " |")
table_md = "\n".join(md_lines)
open("comparison_three_models.md", "w").write(table_md + "\n")
print(table_md)

try:
    from google.colab import files
    files.download("comparison_three_models.csv")
    files.download("comparison_three_models.md")
except Exception as e:
    print(f"\n(saved to {os.getcwd()}; auto-download unavailable: {type(e).__name__})")

---

### Reading the numbers

- **MSE / MAE** are on the `ln(mean RV)` scale. **QLIKE** (Patton 2011) exponentiates
  both sides, so it compares variances directly and is the more robust ranking
  criterion for volatility forecasts — prefer it when the three metrics disagree.
- All models score the **same** test rows (647 / 643 / 626), asserted in section 4,
  so the columns are directly comparable.
- HAR-RV is deterministic OLS — one run, no seed spread. The deep models report the
  mean over `ITR` seeds with `*_std` across them. **If the gap between two models is
  smaller than their seed spread, it is not a result** — the last cell of section 8
  checks exactly that.

### Caveats worth stating before quoting this table

- **One hyper-parameter set for three horizons.** These were supplied as a single
  config and held fixed while only `--pred_len` moved. A horizon-specific search
  (`tune.py`, one study per horizon) would likely change the ranking, and the plain
  and event models could well prefer different settings. This table answers "same
  architecture, events on vs off", not "best vs best".
- **The event model is strictly larger.** `--use_events` adds an embedding and a FiLM
  head, so it is not a like-for-like parameter count. A win could be capacity rather
  than information; the `--event_past` / `--event_future` ablation switches in
  `run.py` are the way to separate those.
- **Future events are a schedule, not an outcome.** The forecast-window event tensor
  encodes only *that a release is scheduled*, never its value — a real property of a
  macro calendar, and section 4 of `smoke_test_events.py` asserts the window starts
  strictly after the look-back ends.
- **HAR-Q is deliberately absent.** It needs realized quarticity, which
  `EURUSD-RV.csv` does not carry, so it runs on a different series and different test
  rows and would not be comparable here.